In [1]:
import chromadb
import requests
from dotenv import load_dotenv
import os
from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction

load_dotenv()

ENV = os.getenv("ENV")
OLLAMA_SERVER = "http://100.109.42.112:11434" if ENV.lower() == "mac" else "http://localhost:11434"

/Users/khaspper/Documents/Projects/Phren/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
ollama_ef = OllamaEmbeddingFunction(
    url=OLLAMA_SERVER,                # note: just the base URL, no /api/embed
    model_name="nomic-embed-text:v1.5",
)

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="my_collection", embedding_function=ollama_ef)

In [3]:
documents = [
  "This is a document about bananas",
  "This is a document about apples"
  ]

So this fucking code... 
documents = [
  "This is a document about bananas",
  "This is a document about apples"
  ]

response = requests.post(
    f"{OLLAMA_SERVER}/api/embed",
    json={
        "model": "nomic-embed-text:v1.5",
        "input": documents
    }
)

print(response.json())

gave me this error

"{'error': 'model "nomic-embed-text:v1.5" not found, try pulling it first'}"

even though it did exist

Problem 1: Connection refused

Ollama by default only listens on localhost it won't accept connections from outside even over Tailscale...
Fix: Configure it to listen on the Tailscale IP via systemd override. hence the "OLLAMA_HOST=100.109.42.112 ollama serve" command to run in the ubuntu server

Problem 2: Two Ollama instances

When we ran OLLAMA_HOST=... ollama serve manually it created a second instance with no models the systemd service (the real one) was still running separately on localhost
Fix: Stop using manual ollama serve configure the systemd service instead

Problem 3: systemd override missing [Service] header

The override file saved without the header, so systemd ignored the environment variables
Fix: Manually nano'd the file and added [Service] on top

Problem 4: Wrong models directory

The systemd service runs as the ollama user, which looks for models in /usr/share/ollama/.ollama/models. But the models were pulled as markus into /home/markus/.ollama/models.
Fix: rsync'd the models to /usr/share/ollama/.ollama/models and dropped the OLLAMA_MODELS override

End state:

Systemd manages Ollama, always running in the background
Listens on the Tailscale IP so Mac can reach it
Models found correctly
From now on: sudo systemctl start/stop/restart ollama

In [4]:
collection.add(
    ids=["id3", "id4"],
    metadatas=[{"source": "my_source", "source_1": "my_source_1"}, {"source": "my_source_0", "source_2": "my_source_2"}],
    documents=documents
)

In [9]:
results = collection.query(
  query_texts=["bananas"],
  n_results=2
)

results

{'ids': [['id3', 'id4']],
 'embeddings': None,
 'documents': [['This is a document about bananas',
   'This is a document about apples']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source_1': 'my_source_1', 'source': 'my_source'},
   {'source': 'my_source_0', 'source_2': 'my_source_2'}]],
 'distances': [[0.23769396543502808, 0.5261155962944031]]}

Alright so just now I just downloaded gemma4:12b so it can read my excalidraw drawings and I can embed that 